In [ ]:
#%pip install paddleocr
#%pip install paddlepaddle==3.2.2

from paddleocr import PaddleOCR as PaddleOCRModel
import re
import csv

from pathlib import Path
import os

# Run once
os.chdir(Path.cwd().parent)
os.chdir(Path.cwd().parent)



## Class for Paddle OCR

In [2]:
class OCRpipeline:
    def __init__(self, text_detection_model_name = "PP-OCRv5_mobile_det", text_recognition_model_name = "PP-OCRv5_mobile_rec", use_doc_orientation_classify = False, use_doc_unwarping = False, use_textline_orientation = False):
        self.ocr_model = PaddleOCRModel(text_detection_model_name=text_detection_model_name,
                                    text_recognition_model_name=text_recognition_model_name,
                                    use_doc_orientation_classify=use_doc_orientation_classify,
                                    use_doc_unwarping=use_doc_unwarping,
                                    use_textline_orientation=use_textline_orientation)
        
    def paddleInference(self, img_path):
        # Input image path, run OCR and return text
        result = self.ocr_model.predict(img_path)
        return result
    

    def order_best_text(self, results, confidence_threshold = 0.5):
        rec_texts = results[0]["rec_texts"]
        rec_scores = results[0]["rec_scores"]

        # Pair text with score
        text_score_pairs = list(zip(rec_texts, rec_scores))

        # Filter out pairs with low confidence
        filtered_pairs = [pair for pair in text_score_pairs if pair[1] >= confidence_threshold]

        # Sort by score (highest first)
        text_score_pairs_sorted = sorted(filtered_pairs, key=lambda x: x[1], reverse=True)

        return text_score_pairs_sorted if text_score_pairs_sorted else None
    

    def extract_best_text(self, text_score_pairs_sorted, pattern = re.compile(r'^\d{4,5}(DL)?$')):
        # Extract the best pattern matching text from the sorted pairs
        pattern = pattern

        if text_score_pairs_sorted is None:
            return "null"
        
        for text, score in text_score_pairs_sorted:
            text = text.strip().upper()
        
            # Must match part-number pattern
            if pattern.match(text):
                return text
            
        return "null"


            




## Run inference on all crops to help with annotations

In [3]:
# Run on test set and write results to CSV
folder_paths = ["data/OCRcropsv3/test/rawCropTest"]

output_files = ["baselineOcrResultsTest.csv"]

# Loop over folders and output files
for folder_path, output_file in zip(folder_paths, output_files):
    self = OCRpipeline()
    folder = Path(folder_path)

    # Write results to CSV
    with open(output_file, "w", newline="") as f:
        writer = csv.writer(f)
        writer.writerow(["filename", "best_text"])

        for img_path in folder.glob("*"):
            ocr_results = self.paddleInference(str(img_path))
            text_score_pairs_sorted = self.order_best_text(ocr_results)
            best_text = self.extract_best_text(text_score_pairs_sorted)
            writer.writerow([img_path.name, best_text])

/home/daniel/Documents/deepLearning/Project2_TrOCR/Project2Directory/Daniel/.venv/lib/python3.12/site-packages/paddle/utils/cpp_extension/extension_utils.py:718: UserWarning: No ccache found. Please be aware that recompiling all source files may be required. You can download and install ccache from: https://github.com/ccache/ccache/blob/master/doc/INSTALL.md
  warnings.warn(warning_message)
Creating model: ('PP-OCRv5_mobile_det', None, None)
Model files already exist. Using cached files. To redownload, please delete the directory manually: `/home/daniel/.paddlex/official_models/PP-OCRv5_mobile_det`.
Creating model: ('PP-OCRv5_mobile_rec', None, None)
Model files already exist. Using cached files. To redownload, please delete the directory manually: `/home/daniel/.paddlex/official_models/PP-OCRv5_mobile_rec`.


In [4]:
# Run on train and val sets to make annotation more efficient
folder_paths = ["data/OCRcrops/train/rawCropTrain",
                "data/OCRcrops/val/rawCropVal"]

output_files = ["ocrResultsTrain.csv",
                "ocrResultsVal.csv"]



# Loop over folders and output files
for folder_path, output_file in zip(folder_paths, output_files):
    self = OCRpipeline()
    folder = Path(folder_path)

    # Write results to CSV
    with open(output_file, "w", newline="") as f:
        writer = csv.writer(f)
        writer.writerow(["filename", "best_text"])

        for img_path in folder.glob("*"):
            ocr_results = self.paddleInference(str(img_path))
            text_score_pairs_sorted = self.order_best_text(ocr_results)
            best_text = self.extract_best_text(text_score_pairs_sorted)
            writer.writerow([img_path.name, best_text])

Creating model: ('PP-OCRv5_mobile_det', None, None)
Model files already exist. Using cached files. To redownload, please delete the directory manually: `/home/daniel/.paddlex/official_models/PP-OCRv5_mobile_det`.
Creating model: ('PP-OCRv5_mobile_rec', None, None)
Model files already exist. Using cached files. To redownload, please delete the directory manually: `/home/daniel/.paddlex/official_models/PP-OCRv5_mobile_rec`.
Creating model: ('PP-OCRv5_mobile_det', None, None)
Model files already exist. Using cached files. To redownload, please delete the directory manually: `/home/daniel/.paddlex/official_models/PP-OCRv5_mobile_det`.
Creating model: ('PP-OCRv5_mobile_rec', None, None)
Model files already exist. Using cached files. To redownload, please delete the directory manually: `/home/daniel/.paddlex/official_models/PP-OCRv5_mobile_rec`.


# Reference labelStudio.ipynb